In [ ]:
%run milp_engine.ipynb

In [ ]:
import pandas as pd
import numpy as np


# ============================================================
# REALISED SCHEDULE EVALUATION
# ============================================================
#
# Purpose:
#   Evaluate a MILP-generated schedule against observed/
#   realised surgical durations.
#
# Pipeline:
#
#   ML predictions
#        ↓
#   MILP schedule
#        ↓
#   Fixed room assignment + planned sequence
#        ↓
#   ACTUAL_DURATION
#        ↓
#   Delay propagation
#        ↓
#   Realised operational performance
#
# ============================================================


# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

TEST_FILE = (
    "new_data/"
    "mover_epic_final_test_features.csv"
)

COHORT_FILE = (
    "Cohort/"
    "fixed_cohort.csv"
)

ROOM_CAPACITY = 480.0

TURNOVER = 20.0

N_ROOMS = 3


# ============================================================
# 2. LOAD ACTUAL TEST DATA
# ============================================================

test_df = pd.read_csv(
    TEST_FILE
)

print("=" * 70)
print("LOADED TEST DATA")
print("=" * 70)

print(
    f"Number of test cases: "
    f"{len(test_df)}"
)

print(
    f"Columns: "
    f"{len(test_df.columns)}"
)


# ============================================================
# 3. LOAD FIXED COHORT
# ============================================================

cohort_df = pd.read_csv(
    COHORT_FILE
)

cohort_ids = (
    cohort_df["LOG_ID"]
    .tolist()
)

N_CASES = len(cohort_ids)

print(
    f"Fixed cohort size: "
    f"{N_CASES}"
)


# ============================================================
# 4. CHECK ACTUAL DURATION COLUMN
# ============================================================

if "ACTUAL_DURATION" not in test_df.columns:

    raise ValueError(
        "ACTUAL_DURATION column was not found "
        "in the test dataset."
    )


# ============================================================
# 5. EXTRACT ACTUAL DURATIONS
# ============================================================

actual_df = test_df[
    test_df["LOG_ID"].isin(
        cohort_ids
    )
][
    [
        "LOG_ID",
        "ACTUAL_DURATION"
    ]
].copy()


# Preserve fixed cohort order

cohort_order = {
    log_id: i
    for i, log_id in enumerate(
        cohort_ids
    )
}


actual_df["Surgery"] = (
    actual_df["LOG_ID"]
    .map(cohort_order)
)


actual_df = (
    actual_df
    .sort_values("Surgery")
    .reset_index(drop=True)
)


# ============================================================
# 6. SAFETY CHECK
# ============================================================

if len(actual_df) != N_CASES:

    raise ValueError(
        f"Cohort mismatch: expected "
        f"{N_CASES} cases but found "
        f"{len(actual_df)} actual-duration records."
    )


if actual_df["ACTUAL_DURATION"].isna().any():

    raise ValueError(
        "Missing ACTUAL_DURATION values "
        "were found in the fixed cohort."
    )


if (
    actual_df["ACTUAL_DURATION"] <= 0
).any():

    raise ValueError(
        "Non-positive ACTUAL_DURATION "
        "values were found."
    )


print("\n")
print("=" * 70)
print("ACTUAL SURGERY DURATIONS")
print("=" * 70)

print(
    actual_df[
        [
            "Surgery",
            "LOG_ID",
            "ACTUAL_DURATION"
        ]
    ].to_string(index=False)
)


# ============================================================
# 7. MILP SCHEDULE INPUT
# ============================================================
#
# IMPORTANT:
#
# These values must come from the VALIDATED MILP solution.
#
# Example:
#   LightGBM
#   λ = 0.5
#   12 cases
#   3 rooms
#   turnover = 20
#
# The MILP schedule itself is NOT changed.
#
# ============================================================


# ------------------------------------------------------------
# Room assignment from validated LightGBM λ=0.5 solution
# ------------------------------------------------------------

room_assignment = {

    0: 1,
    1: 1,
    2: 2,
    3: 0,
    4: 2,
    5: 0,
    6: 2,
    7: 1,
    8: 0,
    9: 1,
    10: 0,
    11: 2,

}


# ------------------------------------------------------------
# Planned start times from validated MILP solution
# ------------------------------------------------------------

start_times = {

    0: 846.87,
    1: 594.56,
    2: 786.63,
    3: 741.46,
    4: 489.52,
    5: 510.00,
    6: 232.32,
    7: 380.00,
    8: 265.55,
    9: 0.00,
    10: 0.00,
    11: 0.00,

}


# ============================================================
# 8. CHECK MILP SCHEDULE STRUCTURE
# ============================================================

if len(room_assignment) != N_CASES:

    raise ValueError(
        "Room assignment does not contain "
        "all cases."
    )


if len(start_times) != N_CASES:

    raise ValueError(
        "Start-time dictionary does not contain "
        "all cases."
    )


for i in range(N_CASES):

    if room_assignment[i] not in range(
        N_ROOMS
    ):

        raise ValueError(
            f"Surgery {i} assigned to "
            f"invalid room."
        )


# ============================================================
# 9. CREATE ACTUAL-DURATION LOOKUP
# ============================================================

actual_map = dict(
    zip(
        actual_df["Surgery"],
        actual_df["ACTUAL_DURATION"]
    )
)


# ============================================================
# 10. RECONSTRUCT CHRONOLOGICAL ROOM SEQUENCES
# ============================================================
#
# The MILP solution determines the planned order.
#
# We keep that order fixed.
#
# Actual duration then determines whether later cases
# have to be delayed.
#
# ============================================================

room_cases = {

    room: []

    for room in range(N_ROOMS)

}


for surgery in range(N_CASES):

    room = room_assignment[
        surgery
    ]

    room_cases[room].append(
        surgery
    )


for room in room_cases:

    room_cases[room].sort(
        key=lambda i:
        start_times[i]
    )


# ============================================================
# 11. REALISED SCHEDULE WITH DELAY PROPAGATION
# ============================================================

realised_schedule = []


for room in range(N_ROOMS):

    cases = room_cases[room]

    previous_finish = None

    for surgery in cases:

        planned_start = float(
            start_times[surgery]
        )

        actual_duration = float(
            actual_map[surgery]
        )


        # ----------------------------------------------------
        # First case
        # ----------------------------------------------------

        if previous_finish is None:

            realised_start = (
                planned_start
            )


        # ----------------------------------------------------
        # Subsequent cases
        # ----------------------------------------------------
        #
        # A case cannot start before:
        #
        # previous actual completion
        # + turnover
        #
        # ----------------------------------------------------

        else:

            realised_start = max(

                planned_start,

                previous_finish
                +
                TURNOVER

            )


        realised_finish = (

            realised_start
            +
            actual_duration

        )


        start_delay = (

            realised_start
            -
            planned_start

        )


        realised_schedule.append({

            "Surgery":
                surgery,

            "LOG_ID":
                cohort_ids[surgery],

            "Room":
                room,

            "Planned_Start":
                planned_start,

            "Realised_Start":
                realised_start,

            "Actual_Duration":
                actual_duration,

            "Realised_Finish":
                realised_finish,

            "Start_Delay":
                start_delay,

        })


        previous_finish = (
            realised_finish
        )


# ============================================================
# 12. CREATE REALISED SCHEDULE DATAFRAME
# ============================================================

realised_df = pd.DataFrame(
    realised_schedule
)


realised_df = (

    realised_df

    .sort_values(
        [
            "Room",
            "Realised_Start"
        ]
    )

    .reset_index(drop=True)

)


# ============================================================
# 13. CHECK FOR REALISED OVERLAPS
# ============================================================

validation_errors = []


for room in range(N_ROOMS):

    room_df = (

        realised_df[
            realised_df["Room"] == room
        ]

        .sort_values(
            "Realised_Start"
        )

    )


    previous_finish = None


    for _, row in room_df.iterrows():

        if previous_finish is not None:

            required_start = (

                previous_finish
                +
                TURNOVER

            )

            if (
                row["Realised_Start"]
                <
                required_start
            ):

                validation_errors.append(

                    f"Room {room}: "
                    f"Surgery "
                    f"{int(row['Surgery'])} "
                    f"starts before the "
                    f"previous case plus "
                    f"turnover."

                )


        previous_finish = (
            row["Realised_Finish"]
        )


# ============================================================
# 14. REALISED ROOM-LEVEL METRICS
# ============================================================

room_results = []


for room in range(N_ROOMS):

    room_df = (

        realised_df[
            realised_df["Room"] == room
        ]

        .sort_values(
            "Realised_Start"
        )

    )


    if len(room_df) == 0:

        finish = 0.0

        busy_time = 0.0

    else:

        finish = room_df[
            "Realised_Finish"
        ].max()

        busy_time = room_df[
            "Actual_Duration"
        ].sum()


    overtime = max(

        finish
        -
        ROOM_CAPACITY,

        0.0

    )


    start_delay_total = (
        room_df[
            "Start_Delay"
        ].sum()
    )


    max_start_delay = (

        room_df[
            "Start_Delay"
        ].max()

        if len(room_df) > 0

        else 0.0

    )


    room_results.append({

        "Room":
            room,

        "Cases":
            room_df[
                "Surgery"
            ].tolist(),

        "Finish":
            finish,

        "Actual_Busy_Time":
            busy_time,

        "Overtime":
            overtime,

        "Start_Delay_Total":
            start_delay_total,

        "Max_Start_Delay":
            max_start_delay,

    })


room_results_df = pd.DataFrame(
    room_results
)


# ============================================================
# 15. SYSTEM-LEVEL METRICS
# ============================================================

realised_makespan = (

    realised_df[
        "Realised_Finish"
    ].max()

)


realised_overtime = (

    room_results_df[
        "Overtime"
    ].sum()

)


total_actual_duration = (

    realised_df[
        "Actual_Duration"
    ].sum()

)


# ------------------------------------------------------------
# Nominal capacity load ratio
# ------------------------------------------------------------

nominal_capacity = (

    N_ROOMS
    *
    ROOM_CAPACITY

)


nominal_capacity_load_ratio = (

    total_actual_duration
    /
    nominal_capacity

)


# ------------------------------------------------------------
# Realised-horizon utilisation
# ------------------------------------------------------------

realised_horizon_capacity = (

    N_ROOMS
    *
    realised_makespan

)


realised_horizon_utilisation = (

    total_actual_duration
    /
    realised_horizon_capacity

)


# ============================================================
# 16. REALISED OBJECTIVE
# ============================================================
#
# We use the same objective structure as the MILP:
#
#   OT
#   + 0.25 * Idle
#   + 0.5 * Makespan
#
# Since this realised evaluation uses a realised schedule,
# idle time is not defined as "capacity - finish".
#
# Instead, the primary realised objective is based on:
#
#   Overtime + 0.5 * Makespan
#
# This should be clearly labelled as a realised evaluation
# metric rather than the original MILP objective.
#
# ============================================================

realised_objective = (

    realised_overtime
    +
    0.5
    *
    realised_makespan

)


# ============================================================
# 17. PRINT CASE-LEVEL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("REALISED SCHEDULE WITH DELAY PROPAGATION")
print("=" * 70)

print(

    realised_df[
        [
            "Surgery",
            "LOG_ID",
            "Room",
            "Planned_Start",
            "Realised_Start",
            "Actual_Duration",
            "Realised_Finish",
            "Start_Delay"
        ]
    ]

    .to_string(index=False)

)


# ============================================================
# 18. PRINT ROOM-LEVEL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("ROOM-LEVEL REALISED RESULTS")
print("=" * 70)


for _, row in room_results_df.iterrows():

    print(

        f"Room {int(row['Room'])}: "

        f"Cases={row['Cases']}, "

        f"Finish={row['Finish']:.2f}, "

        f"Busy={row['Actual_Busy_Time']:.2f}, "

        f"OT={row['Overtime']:.2f}, "

        f"Total Delay="
        f"{row['Start_Delay_Total']:.2f}"

    )


# ============================================================
# 19. PRINT SYSTEM-LEVEL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("REALISED SYSTEM PERFORMANCE")
print("=" * 70)


print(

    f"Realised Makespan: "
    f"{realised_makespan:.2f}"

)


print(

    f"Realised Overtime: "
    f"{realised_overtime:.2f}"

)


print(

    f"Total Actual Surgery Time: "
    f"{total_actual_duration:.2f}"

)


print(

    f"Nominal Capacity Load Ratio: "
    f"{nominal_capacity_load_ratio:.4f}"

)


print(

    f"Realised-Horizon Utilisation: "
    f"{realised_horizon_utilisation:.4f}"

)


print(

    f"Realised Evaluation Objective: "
    f"{realised_objective:.2f}"

)


# ============================================================
# 20. VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("REALISED SCHEDULE VALIDATION")
print("=" * 70)


if len(validation_errors) == 0:

    print(
        "PASS: No realised overlap or "
        "turnover violations detected."
    )

else:

    print(
        f"FAIL: {len(validation_errors)} "
        f"validation errors."
    )

    for error in validation_errors:

        print(
            "ERROR:",
            error
        )


# ============================================================
# 21. SAVE RESULTS
# ============================================================

OUTPUT_SCHEDULE = (
    "Results/"
    "realised_lightgbm_lambda05_schedule.csv"
)

OUTPUT_ROOMS = (
    "Results/"
    "realised_lightgbm_lambda05_room_results.csv"
)

OUTPUT_SUMMARY = (
    "Results/"
    "realised_lightgbm_lambda05_summary.csv"
)


realised_df.to_csv(

    OUTPUT_SCHEDULE,

    index=False

)


room_results_df.to_csv(

    OUTPUT_ROOMS,

    index=False

)


summary_df = pd.DataFrame({

    "Model":
        ["LightGBM"],

    "Lambda":
        [0.5],

    "Cases":
        [N_CASES],

    "Realised_Makespan":
        [realised_makespan],

    "Realised_Overtime":
        [realised_overtime],

    "Total_Actual_Duration":
        [total_actual_duration],

    "Nominal_Capacity_Load_Ratio":
        [nominal_capacity_load_ratio],

    "Realised_Horizon_Utilisation":
        [realised_horizon_utilisation],

    "Realised_Evaluation_Objective":
        [realised_objective],

    "Validation_Passed":
        [len(validation_errors) == 0]

})


summary_df.to_csv(

    OUTPUT_SUMMARY,

    index=False

)


# ============================================================
# 22. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print(
    OUTPUT_SCHEDULE
)

print(
    OUTPUT_ROOMS
)

print(
    OUTPUT_SUMMARY
)